In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
# Exploratory Data Analysis

# Objective
# Understand the structure, quality, and patterns within the hospital readmission dataset before building a machine learning model.

# See if the data makes sense
df = pd.read_csv("../data/raw/diabetic_data.csv")
print(df.head())
print(df.shape)


"""
The dataset contains 101,766 hospital encounters and 50 variables.
Each row represents one patient encounter.
Each column is a piece of information about that encounter
"""




In [ ]:
# We need to see if all the columns here match what is on the website and CSV.
print(df.columns)


In [ ]:
# Helpful for classifying predictors as either numerical or categorical.
print(df.info())

"""
    For the columns
    Numerical features (ints): 13 columns (model can process these, these are already numbers)
    Objects (categorical): 37 columns (model cannot understand this, needs and encoding)
"""

"""
    Identification colums like encounter_id and patient_nbr are not medical information.
    They are not useful in determining the health of the patient.
    They can be discarded to avoid the model learning useless patterns.
"""

In [ ]:
## Missing Values
# Weight, payer_code, medical_specialty may contain missing values.

# many entries may have ? which is technically not empty so python will spit out non-null

(df == "?").sum().sort_values(ascending=False)

"""
    Results show that there is too much missing data in weight, the model cannot learn if only < 5% of patients have data for this feature.
    Too much missing data reduces reliability and introduces bias. We will remove this

    as for medical_specialty, the possibilities of missing data is because:
        - This specialty was not recorded
        - The patient did not see a specialist
        - encounter type did not require specialty information
        - maybe seeing a different specialist have different readmission patterns. 
        - (we can keep this for now)
    
    The largest missing areas occur in:
    - weight (~97%)
    - medical_specialty (~49%)
    - payer_code (~40%) -> insurance type: (medicaid, medicare, ...)

    The weight feature contains too many missing values and may be removed during preprocessing. Medical specialty and payer code require additional analysis because they may contain useful predictive information despite missing values.

    Diagnosis variables contain minimal missing values and will likely be retained.
"""



In [ ]:
## Target Variable Anaylsis

# How many patients were actually readmitted?
# Goal: Analyze redistribution of target variables

# actual counts:
print("Actual counts: ")
print(df["readmitted"].value_counts())

# percentage
print("\nPercentages: ")
print(df["readmitted"].value_counts(normalize=True) * 100)


"""
You are trying to identify risks of being readmitted in <30 days.
The class you care about is the smallest one.

There is an imbalance in the dataset for the target variable.
Classes are not evenly distributed.
Classes are: <30, >30, NO

where currently we have:
<30 -> Positive
>30 -> Negative
No  -> Negative

We may change to (encoding):
1 = readmitted within 30 days
0 = not readmitted within 30 days

"""

In [ ]:
pd.crosstab(
    df["age"],
    df["readmitted"],
    normalize="index"
) * 100

# ages 20-30 have the highest rate of readmission under 30.
# Children 0-10 are unlikely to be readmitted within 30 days or ever.
# Patients age 70-90 have the highest proportion of readmission after 30 days.


"""
Younger patients from ages 0-10 years have a lower rate of readmission within 30 days compared to adult patients. Among adult patients, the percentage of patients readmitted within 30 days stays roughly consistent (11-14%). This suggests that age may contribute useful information to the prediction model, although it is unlikely to be a strong predictor on its own.
"""

In [ ]:
import matplotlib.pyplot as plt

age_readmission = pd.crosstab(
    df["age"], 
    df["readmitted"]
)
age_readmission.plot(
    kind="bar",
    figsize=(10, 5)
)

plt.title("Readmission by Age Group")
plt.ylabel("Number of Patients")
plt.xlabel("Age Group")
plt.xticks(rotation=45)
plt.show()

In [ ]:
df.groupby("readmitted")[
    [
        "time_in_hospital",
        "num_medications",
        "number_inpatient",
        "number_emergency",
        "number_outpatient"
    ]
].mean()

"""

number_inpatients is more than 3 times higher for patients readmitted within 30 days.

The more emergency visits means a higher likelyhood of readmission.

Patients who are taking more medications tend to have more complex medical conditions so number of medications is highest for readmission within 30 days.

time in hospital would increase for people readmitted within 30 days.

Number of outpatience is going to be lower for patients readmitted within 30 days because these patients need more care.

"""

In [ ]:
pd.crosstab(
    df["diabetesMed"],
    df["readmitted"],
    normalize="index"
) * 100

""" 
patients who were readmitted within 30 days were more likely to get diabetes medications. Only patients who were never readmitted are more likely to not get diabetes medication.
"""

In [ ]:
pd.crosstab(
    df["insulin"],
    df["readmitted"],
    normalize="index"
) * 100

""" 
patients with "DOWN"(decreased) and "UP"(increased) insulin had higher proportions of readmission within 30 days than patients with "No"(not prescribed) or "Steady" (unchanged) insulin.
"""

In [ ]:
pd.crosstab(df["insulin"], df["readmitted"])

In [ ]:
pd.crosstab(df["A1Cresult"], df["readmitted"], normalize="index") * 100

"""
The distribution of readmission outcomes is very similar across A1C result categories. Patients with elevated A1C (>7, >8) levels do not appear to be that different from patients with normal A1C results. A1C along does not appear to be a strong individual predictor. 
"""